# NB-01 — めぐ指数: データ探索・前処理

**目的**: めぐ指数の統合回帰に使用する4つのデータソース（走破タイム・スプリット・斤量・フィールド質）の品質確認と外れ値除去基準の決定

**入力**: `race_result`, `race_result_lap`, `race_shutuba`, `horse_result`  
**出力**: クリーニング済みデータセット (`megu_dataset.parquet`)、外れ値除去基準の確定

**実行順序**: このノートブックを最初に実行する。NB-02〜NB-06 はこのノートブックの出力に依存する。

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/keiba-vpn')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = Path('output/nb01')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Setup OK')

## 1. データ読み込み

In [ ]:
from src.db.session import get_session, init_engine
from sqlalchemy import text

init_engine()

# --- 走破タイム・レース基本情報 ---
with get_session() as session:
    df_result = pd.read_sql(text("""
        SELECT
            rr.race_id,
            rr.horse_id,
            rr.finish_pos,
            rr.finish_time_sec,
            rr.weight_carried,
            rr.horse_weight,
            r.distance,
            r.surface,
            r.course,
            r.track_condition,
            r.race_date,
            r.grade,
            r.race_type,
            h.sex,
            h.is_imported
        FROM race_results rr
        JOIN races r ON rr.race_id = r.race_id
        LEFT JOIN horses h ON rr.horse_id = h.horse_id
        WHERE r.surface IN ('芝', 'ダート')
          AND rr.finish_time_sec IS NOT NULL
          AND rr.finish_time_sec > 0
          AND r.race_date >= '2018-01-01'
    """), session.bind)

# is_imported 欠損は False として扱う
df_result['is_imported'] = df_result['is_imported'].fillna(False)

print(f'race_results: {len(df_result):,} 行')
df_result.head(3)

In [ ]:
# --- 前半スプリットタイム ---
# race_result_lap から距離の50%地点に最も近いスプリットを取得
with get_session() as session:
    df_lap = pd.read_sql(text("""
        SELECT
            rl.race_id,
            rl.split_distance_m,
            rl.split_time_sec
        FROM race_result_lap rl
        WHERE rl.split_time_sec IS NOT NULL
    """), session.bind)

print(f'race_result_lap: {len(df_lap):,} 行')
print(f'ユニークrace_id: {df_lap["race_id"].nunique():,}')
print(f'\nsplit_distance_m の分布:')
print(df_lap['split_distance_m'].value_counts().head(20))

In [ ]:
# --- 斤量データ（race_shutuba から取得） ---
with get_session() as session:
    df_weight = pd.read_sql(text("""
        SELECT
            e.race_id,
            e.horse_id,
            e.weight_carried
        FROM entries e
        WHERE e.weight_carried IS NOT NULL
    """), session.bind)

print(f'entries (weight): {len(df_weight):,} 行')
print(f'\n斤量の基本統計:')
print(df_weight['weight_carried'].describe())

In [ ]:
# --- 獲得賞金履歴（horse_result から FQ 計算用） ---
with get_session() as session:
    df_earnings = pd.read_sql(text("""
        SELECT
            hr.horse_id,
            hr.total_earnings,
            hr.as_of_race_id
        FROM horse_stats_snapshot hr
        WHERE hr.total_earnings IS NOT NULL
    """), session.bind)

# total_earnings が存在しない場合は horse_result JSON から取得する代替手段を検討
print(f'horse_stats_snapshot (earnings): {len(df_earnings):,} 行')
print(f'\n獲得賞金の基本統計（円）:')
print(df_earnings['total_earnings'].describe())

## 2. 走破タイムの分布確認

In [ ]:
# 距離×馬場種別ごとの走破タイム分布
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

distance_bins = [1000, 1200, 1400, 1600, 1800, 2000, 2200, 2400, 4000]
df_result['distance_band'] = pd.cut(df_result['distance'], bins=distance_bins, right=False)

for ax, (surface, group) in zip(axes.flat, df_result.groupby(['surface', 'distance_band'])):
    group['finish_time_sec'].hist(bins=50, ax=ax, color='steelblue', alpha=0.7)
    ax.set_title(f'{surface[0]} {surface[1]}', fontsize=9)
    ax.set_xlabel('走破タイム（秒）', fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'finish_time_distribution.png', dpi=120)
plt.show()
print('走破タイム分布: 出力完了')

In [ ]:
# 外れ値検出: 各 (distance, surface, track_condition) セル内での z-score
grp_cols = ['distance_band', 'surface', 'track_condition']
df_result['time_mean'] = df_result.groupby(grp_cols)['finish_time_sec'].transform('mean')
df_result['time_std']  = df_result.groupby(grp_cols)['finish_time_sec'].transform('std')
df_result['time_z']    = (df_result['finish_time_sec'] - df_result['time_mean']) / df_result['time_std']

print('=== z-score > 3 の外れ値件数 ===')
outliers = df_result[df_result['time_z'].abs() > 3]
print(f'外れ値: {len(outliers):,} 件 ({len(outliers)/len(df_result)*100:.2f}%)')
print(outliers[grp_cols + ['finish_time_sec', 'time_mean', 'time_z']].head(10))

## 3. スプリットデータの利用可能性確認

In [ ]:
# 距離の50%以下で最近傍のスプリット点を選択する関数
def select_split_point(distance: int, available_splits: list[int]) -> int | None:
    """レース距離の50%以下で最近傍のスプリット点を返す"""
    target = distance * 0.5
    candidates = [s for s in sorted(available_splits) if s <= target]
    return max(candidates) if candidates else None

# 各レースの利用可能スプリット点を確認
df_splits_available = df_lap.groupby('race_id')['split_distance_m'].apply(list).reset_index()
df_splits_available.columns = ['race_id', 'available_splits']

df_check = df_result[['race_id', 'distance']].drop_duplicates().merge(
    df_splits_available, on='race_id', how='left'
)
df_check['available_splits'] = df_check['available_splits'].apply(lambda x: x if isinstance(x, list) else [])
df_check['selected_split'] = df_check.apply(
    lambda r: select_split_point(r['distance'], r['available_splits']), axis=1
)

coverage = df_check['selected_split'].notna().mean()
print(f'スプリット取得カバレッジ: {coverage*100:.1f}%')
print(f'欠損レース数: {df_check["selected_split"].isna().sum():,}')
print(f'\n選択スプリット点の分布:')
print(df_check['selected_split'].value_counts().head(15))

## 4. フィールド質（FQ）の算出

In [ ]:
# FQ = 上位5着以内の馬の獲得賞金履歴平均（テンポラルリーク防止: as_of_race_id 以前のみ）
# 【確定 U-3】外れ値処理:
#   1. 新馬戦（race_type='新馬'）または全馬total_earnings=0 → skip_fq フラグを立て log_fq_dev=0 扱いにする
#   2. 輸入馬（is_imported=True）かつ total_earnings=0 → FQ 計算から除外
#   3. FQ < 1,000,000円 → 1,000,000円フロアクリップ

# 新馬戦レースの race_id を取得
shinme_race_ids = set(df_result[df_result['race_type'] == '新馬']['race_id'].unique())
print(f'新馬戦レース数: {len(shinme_race_ids):,}')

df_top5 = df_result[df_result['finish_pos'] <= 5].copy()

# as_of_race_id に対応するスナップショットから total_earnings を取得
df_fq_raw = df_top5.merge(
    df_earnings[['horse_id', 'as_of_race_id', 'total_earnings']],
    left_on=['horse_id', 'race_id'],
    right_on=['horse_id', 'as_of_race_id'],
    how='left'
)

# 輸入馬かつ total_earnings=0 を FQ 計算から除外
df_fq_raw['exclude_fq'] = (
    df_fq_raw['is_imported'].fillna(False) &
    (df_fq_raw['total_earnings'].fillna(0) == 0)
)
df_fq_valid = df_fq_raw[~df_fq_raw['exclude_fq']].copy()

# レースごとに FQ 算出（有効な馬のみで平均）
fq_by_race = df_fq_valid.groupby('race_id')['total_earnings'].agg(['mean', 'count']).reset_index()
fq_by_race.columns = ['race_id', 'fq', 'fq_n']

# 新馬戦・全馬0賞金レースは fq_skip フラグ
all_zero_race_ids = set(
    df_fq_raw.groupby('race_id')['total_earnings']
    .apply(lambda x: (x.fillna(0) == 0).all())
    .pipe(lambda s: s[s].index)
)
fq_skip_race_ids = shinme_race_ids | all_zero_race_ids
fq_by_race['fq_skip'] = fq_by_race['race_id'].isin(fq_skip_race_ids)

# FQ フロアクリップ: 100万円未満を 100万円に置換（[U-3] 決定値）
FQ_FLOOR = 1_000_000
fq_by_race['fq_clipped'] = fq_by_race['fq'].clip(lower=FQ_FLOOR)
fq_by_race.loc[fq_by_race['fq_skip'], 'fq_clipped'] = np.nan  # skip 対象は NaN のまま

# log(FQ): skip 対象は NaN → NB-02 で log_fq_dev=0 に変換される
fq_by_race['log_fq'] = np.where(
    fq_by_race['fq_skip'],
    np.nan,
    np.log(fq_by_race['fq_clipped'])
)

print('FQ の基本統計（フロアクリップ後, 円）:')
print(fq_by_race[~fq_by_race['fq_skip']]['fq_clipped'].describe())
print(f'\nFQ 欠損レース（全馬0賞金/新馬戦 → Δlevel=0）: {fq_by_race["fq_skip"].sum():,}')
print(f'FQ < {FQ_FLOOR:,}円 のクリップ件数: {(fq_by_race["fq"] < FQ_FLOOR).sum():,}')

# log(FQ) の分布確認
fq_by_race['log_fq'].dropna().hist(bins=60, color='coral', alpha=0.8)
plt.xlabel('log(FQ)')
plt.title(f'フィールド質 log(FQ) 分布（フロア={FQ_FLOOR:,}円）')
plt.savefig(OUTPUT_DIR / 'fq_distribution.png', dpi=120)
plt.show()

In [ ]:
# par_FQ: 全レース log(FQ) の平均（fq_skip 対象を除外して算出）
par_log_fq = fq_by_race.loc[~fq_by_race['fq_skip'], 'log_fq'].mean()
print(f'par_FQ（幾何平均, フロアクリップ後）: {np.exp(par_log_fq):,.0f} 円')
print(f'log(par_FQ): {par_log_fq:.4f}')
print(f'  ※ 新馬戦・全馬0賞金レース ({len(fq_skip_race_ids):,} 件) は par_FQ 算出から除外')

## 5. データ結合とクリーニング

In [ ]:
from src.analysis.track_speed import get_tsi_offset  # TrackSpeedIndex 取得（既存実装）

# 1) スプリットタイムをレースにマージ
df_split_selected = (
    df_check[['race_id', 'distance', 'selected_split']]
    .merge(df_lap, left_on=['race_id', 'selected_split'], right_on=['race_id', 'split_distance_m'], how='left')
)

# 2) 斤量をマージ
df_main = df_result.merge(
    df_weight[['race_id', 'horse_id', 'weight_carried']].rename(columns={'weight_carried': 'weight_entry'}),
    on=['race_id', 'horse_id'], how='left'
)

# 3) FQ をマージ（log_fq + fq_skip フラグを含める）
df_main = df_main.merge(
    fq_by_race[['race_id', 'fq', 'log_fq', 'fq_skip']],
    on='race_id', how='left'
)
# fq_skip が NaN（レースがFQテーブルに存在しない）は False に
df_main['fq_skip'] = df_main['fq_skip'].fillna(False)

# 4) スプリットをマージ
df_main = df_main.merge(
    df_split_selected[['race_id', 'selected_split', 'split_time_sec']],
    on='race_id', how='left'
)

# 5) 外れ値除去（z-score > 3）
df_main = df_main[df_main['time_z'].abs() <= 3].copy()

# 6) 必須カラムの欠損除去
required_cols = ['finish_time_sec', 'distance', 'surface', 'track_condition', 'weight_entry']
df_clean = df_main.dropna(subset=required_cols).copy()

print(f'クリーニング後: {len(df_clean):,} 行')
print(f'スプリットあり: {df_clean["split_time_sec"].notna().sum():,} 行 ({df_clean["split_time_sec"].notna().mean()*100:.1f}%)')
print(f'FQあり（fq_skip=False）: {(~df_clean["fq_skip"] & df_clean["log_fq"].notna()).sum():,} 行')
print(f'FQ スキップ（Δlevel=0）: {df_clean["fq_skip"].sum():,} 行')

## 6. 欠損・外れ値まとめ

In [ ]:
print('=== データ品質サマリー ===')
summary = {
    '総レコード数（走破タイムあり）': len(df_result),
    '走破タイム外れ値（z>3）除去数': int((df_result['time_z'].abs() > 3).sum()),
    'スプリットタイム カバレッジ': f"{df_clean['split_time_sec'].notna().mean()*100:.1f}%",
    'FQ カバレッジ': f"{df_clean['fq'].notna().sum():,} 行 ({df_clean['fq'].notna().mean()*100:.1f}%)",
    '最終クリーニング後レコード数': len(df_clean),
}
for k, v in summary.items():
    print(f'  {k}: {v}')

print('\n=== 確定した外れ値除去基準 ===')
print('  走破タイム: セル内 z-score > 3 を除去')
print('  FQ 外れ値処理 [U-3]:')
print('    - 新馬戦 / 全馬0賞金レース → Δlevel = 0（fq_skip=True）')
print('    - 輸入馬かつ total_earnings=0 → FQ 算出から除外')
print(f'    - FQ < 1,000,000円 → 1,000,000円フロアクリップ')
print('  FQ 欠損（スナップショット未取得）: log_fq_dev = 0 として処理')
print('  スプリット欠損: Δpace = 0 として処理')
print('  障害レース: surface フィルタで除外済み')

In [ ]:
# クリーニング済みデータを保存（NB-02 以降で使用）
df_clean.to_parquet(OUTPUT_DIR / 'megu_dataset.parquet', index=False)
print(f'保存完了: {OUTPUT_DIR / "megu_dataset.parquet"}')